# BiTro Bulk Pipeline (Notebook)

This notebook runs the bulk pipeline with demo_data paths: HoverNet segmentation -> feature extraction -> split -> clustering -> graph construction -> training.
Edit the config cell first, then run cell by cell.


Before running this notebook, make sure the environment is ready and HoverNet outputs exist in `demo_data/hovernet_output` (see the expected layout in the config cell).


In [1]:
import os
import subprocess
from pathlib import Path

def run_cmd(cmd, env=None, cwd=None):
    print(f"\n[RUN] {' '.join(cmd)}")
    result = subprocess.run(cmd, env=env, cwd=cwd, text=True, check=False)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result


In [2]:
# =========================
# Editable configuration (demo_data)
# =========================
PROJECT_ROOT = Path('.').resolve()
DEMO_ROOT = PROJECT_ROOT / 'demo_data'

# HoverNet output (generate before feature extraction)
# Expected layout:
#   hovernet_output/{wsi_name}/*.png
#   hovernet_output/{wsi_name}/json/*.json
#   hovernet_output/{wsi_name}/mat/*.mat
HOVERNET_OUTPUT_DIR = DEMO_ROOT / 'hovernet_output'
IMAGE_FOLDER = DEMO_ROOT / 'output_patches'
JSON_FOLDER = HOVERNET_OUTPUT_DIR
MAT_FOLDER = HOVERNET_OUTPUT_DIR

# Feature extraction outputs (all features before split)
FEATURE_ALL_DIR = DEMO_ROOT / 'Feature' / 'Bulk' / 'all'

# Train/test split outputs
TRAIN_FEATURE_DIR = DEMO_ROOT / 'Feature' / 'Bulk' / 'train'
TEST_FEATURE_DIR = DEMO_ROOT / 'Feature' / 'Bulk' / 'test'

# Bulk expression and patch/WSI inputs for graph construction
BULK_CSV_PATH = DEMO_ROOT / 'bulk' / 'tpm-TCGA-BRCA-1000-million.csv'
PATCHES_DIR = DEMO_ROOT / 'output_patches'
WSI_INPUT_DIR = DEMO_ROOT / 'WSI'

GRAPH_OUTPUT_DIR = DEMO_ROOT / 'Graphs' / 'Bulk'
CHECKPOINT_DIR = DEMO_ROOT / 'checkpoints' / 'bulk'

# KMeans cluster label settings for parquet files
N_CLUSTERS = 7
CLUSTER_RANDOM_STATE = 42

# Feature extraction settings
PCA_COMPONENTS = 128
CHUNK_SIZE = 100

# Graph settings
INTRA_PATCH_DISTANCE_THRESHOLD = 256
INTER_PATCH_K = 8
MAX_CELLS_PER_PATCH = None
MAX_TRAIN_SLIDES = 200
MAX_TEST_SLIDES = 50

# Training settings
GENE_LIST_FILE = DEMO_ROOT / 'Gene' / 'BRCA.txt'
FEATURES_TSV_FILE = DEMO_ROOT / 'bulk' / 'features.tsv'
TPM_CSV_FILE = DEMO_ROOT / 'bulk' / 'tpm-TCGA-BRCA-1000-million.csv'

BATCH_SIZE = 2
GRAPH_BATCH_SIZE = 128
NUM_EPOCHS = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

USE_LORA = True
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = /data/yujk/hovernet2feature/BiTro


In [3]:
required_paths = [
    PROJECT_ROOT / 'utils' / 'extract_bulk_features_dinov3.py',
    PROJECT_ROOT / 'utils' / 'split_features.py',
    PROJECT_ROOT / 'utils' / 'cluster_parquet_features.py',
    PROJECT_ROOT / 'utils' / 'bulk_graph_construction.py',
    PROJECT_ROOT / 'bulk_model' / 'train.py',
    PROJECT_ROOT / 'bulk_model' / 'evaluate.py',
]
for p in required_paths:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")

for d in [HOVERNET_OUTPUT_DIR, FEATURE_ALL_DIR, TRAIN_FEATURE_DIR, TEST_FEATURE_DIR, GRAPH_OUTPUT_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Config check done')


/data/yujk/hovernet2feature/BiTro/utils/extract_bulk_features_dinov3.py: OK
/data/yujk/hovernet2feature/BiTro/utils/split_features.py: OK
/data/yujk/hovernet2feature/BiTro/utils/cluster_parquet_features.py: OK
/data/yujk/hovernet2feature/BiTro/utils/bulk_graph_construction.py: OK
/data/yujk/hovernet2feature/BiTro/bulk_model/train.py: OK
Config check done


In [4]:
# Step 0: extract patches before HoverNet segmentation
run_cmd(['python', str(PROJECT_ROOT / 'utils' / 'extract_patches.py')], cwd=str(PROJECT_ROOT))



[RUN] python /data/yujk/hovernet2feature/BiTro/utils/extract_patches.py

--- Processing TCGA-AN-A04A-01A-01-BS1.647f0482-49a8-4794-b9c4-5941b14fd1af.svs ---
Slide name: TCGA-AN-A04A-01A-01-BS1.647f0482-49a8-4794-b9c4-5941b14fd1af
Magnification: 40
Setting tile prefix to: TCGA-AN-A04A-01A-01-BS1.647f0482-49a8-4794-b9c4-5941b14fd1af_patch_
Extracting tiles for TCGA-AN-A04A-01A-01-BS1.647f0482-49a8-4794-b9c4-5941b14fd1af.svs...
Finished extracting tiles for TCGA-AN-A04A-01A-01-BS1.647f0482-49a8-4794-b9c4-5941b14fd1af.svs. Total patches: 856

--- Processing TCGA-A8-A06U-01A-01-BS1.1182603b-20e7-4a0e-a686-055841cf5ca4.svs ---
Slide name: TCGA-A8-A06U-01A-01-BS1.1182603b-20e7-4a0e-a686-055841cf5ca4
Magnification: 40
Setting tile prefix to: TCGA-A8-A06U-01A-01-BS1.1182603b-20e7-4a0e-a686-055841cf5ca4_patch_
Extracting tiles for TCGA-A8-A06U-01A-01-BS1.1182603b-20e7-4a0e-a686-055841cf5ca4.svs...
Finished extracting tiles for TCGA-A8-A06U-01A-01-BS1.1182603b-20e7-4a0e-a686-055841cf5ca4.svs. To

CompletedProcess(args=['python', '/data/yujk/hovernet2feature/BiTro/utils/extract_patches.py'], returncode=0)

In [4]:
# Before Step 1: run HoVer-Net WSI inference (produces demo_data/hovernet_output)
run_cmd(['bash', str(PROJECT_ROOT.parent / 'hovernet-with-feature-extract' / 'run_wsi.sh')], cwd=str(PROJECT_ROOT.parent / 'hovernet-with-feature-extract'))



[RUN] bash /data/yujk/hovernet2feature/hovernet-with-feature-extract/run_wsi.sh
[INFO] If you have not cloned HoVer-Net yet, run:
       git clone https://github.com/vqdang/hover_net.git
[INFO] Then run this script from:
       /data/yujk/hovernet2feature/hovernet-with-feature-extract

Scanning WSI files...
Found 10 WSI files

Counting processing status...
========== Status Summary ==========
Total files: 10
Processed: 0
Unprocessed: 10

Unprocessed files:
  ○ TCGA-AN-A04A-01A-01-BS1.647f0482-49a8-4794-b9c4-5941b14fd1af.svs
  ○ TCGA-A8-A06U-01A-01-BS1.1182603b-20e7-4a0e-a686-055841cf5ca4.svs
  ○ TCGA-A8-A09I-01A-01-BS1.b0debb62-86b0-459f-b62b-604bb9f17e5d.svs
  ○ TCGA-E2-A14P-01A-03-TSC.e301fe1e-c75b-4ed2-b304-16a9d63fd835.svs
  ○ TCGA-A8-A091-01A-01-BS1.e7e9318d-a627-472a-9f2e-10cc917036d3.svs
  ○ TCGA-A7-A0CD-01A-01-TSA.8966c3d7-c0d2-47a4-badf-df9412a1e5f0.svs
  ○ TCGA-A7-A0CD-01Z-00-DX1.F045B9C8-049C-41BF-8432-EF89F236D34D.svs
  ○ TCGA-AN-A04A-01A-02-BS2.44355b22-95d3-491c-af4c-272

|2026-03-11|20:51:15.140| [INFO] .... Detect #GPUS: 1
|2026-03-11|20:51:23.831| [INFO] ................ Process: TCGA-AN-A04A-01A-01-BS1.647f0482-49a8-4794-b9c4-5941b14fd1af
|2026-03-11|20:51:23.837| [INFO] ............ WARNING: No mask found, generating mask via thresholding at 1.25x!
|2026-03-11|20:51:25.495| [INFO] ........ Preparing Input Output Placement: 1.6644760300405324
Process Chunk 34/36: 100%|########################| 2/2 [00:10<00:00,  5.14s/it]
|2026-03-11|21:02:17.305| [INFO] ........ Inference Time: 651.8096400150098
Post Proc Phase 3: 100%|######################| 151/151 [00:05<00:00, 26.50it/s]
|2026-03-11|21:03:52.018| [INFO] ........ Total Post Proc Time: 94.71202972903848
|2026-03-11|21:04:11.192| [INFO] ........ Save Time: 19.17340353410691
|2026-03-11|21:04:11.192| [INFO] ................ Finish


Failed: TCGA-AN-A04A-01A-01-BS1.647f0482-49a8-4794-b9c4-5941b14fd1af.svs
Progress: 0/10
---
Processing file (1/10): TCGA-A8-A06U-01A-01-BS1.1182603b-20e7-4a0e-a686-055841cf5ca4.svs


|2026-03-11|21:04:17.407| [INFO] .... Detect #GPUS: 1
|2026-03-11|21:04:26.055| [INFO] ................ Process: TCGA-A8-A06U-01A-01-BS1.1182603b-20e7-4a0e-a686-055841cf5ca4
|2026-03-11|21:04:26.062| [INFO] ............ WARNING: No mask found, generating mask via thresholding at 1.25x!
|2026-03-11|21:04:27.672| [INFO] ........ Preparing Input Output Placement: 1.6165047050453722
Process Chunk 25/27: 100%|######################| 21/21 [00:34<00:00,  1.63s/it]
|2026-03-11|21:14:27.534| [INFO] ........ Inference Time: 599.8619667738676
Post Proc Phase 3: 100%|######################| 134/134 [00:04<00:00, 30.72it/s]
|2026-03-11|21:15:18.351| [INFO] ........ Total Post Proc Time: 50.81618946278468
|2026-03-11|21:15:26.374| [INFO] ........ Save Time: 8.022885355167091
|2026-03-11|21:15:26.375| [INFO] ................ Finish


Failed: TCGA-A8-A06U-01A-01-BS1.1182603b-20e7-4a0e-a686-055841cf5ca4.svs
Progress: 0/10
---
Processing file (1/10): TCGA-A8-A09I-01A-01-BS1.b0debb62-86b0-459f-b62b-604bb9f17e5d.svs


...

|2026-03-11|23:28:26.166| [INFO] .... Detect #GPUS: 1
|2026-03-11|23:28:35.266| [INFO] ................ Process: TCGA-BH-A18H-01A-01-TSA.75dba5a3-f9f5-4ff4-814c-7d2451820e03
|2026-03-11|23:28:35.272| [INFO] ............ WARNING: No mask found, generating mask via thresholding at 1.25x!
|2026-03-11|23:28:37.009| [INFO] ........ Preparing Input Output Placement: 1.7426049220375717
Process Chunk 29/36: 100%|######################| 10/10 [00:23<00:00,  2.36s/it]
|2026-03-11|23:40:16.459| [INFO] ........ Inference Time: 699.4499135548249
Post Proc Phase 3: 100%|######################| 158/158 [00:06<00:00, 24.94it/s]
|2026-03-11|23:41:57.295| [INFO] ........ Total Post Proc Time: 100.83539804816246
|2026-03-11|23:42:15.692| [INFO] ........ Save Time: 18.39591035526246
|2026-03-11|23:42:15.692| [INFO] ................ Finish


Failed: TCGA-BH-A18H-01A-01-TSA.75dba5a3-f9f5-4ff4-814c-7d2451820e03.svs
Progress: 0/10
---
Processing file (1/10): TCGA-AO-A0JI-01A-02-BSB.bfccffbb-dc47-4103-beb9-7799d50af784.svs


|2026-03-11|23:42:23.451| [INFO] .... Detect #GPUS: 1
|2026-03-11|23:42:32.731| [INFO] ................ Process: TCGA-AO-A0JI-01A-02-BSB.bfccffbb-dc47-4103-beb9-7799d50af784
|2026-03-11|23:42:32.736| [INFO] ............ WARNING: No mask found, generating mask via thresholding at 1.25x!
|2026-03-11|23:42:37.293| [INFO] ........ Preparing Input Output Placement: 4.561084478162229
Process Chunk 55/60: 100%|########################| 4/4 [00:15<00:00,  3.80s/it]
|2026-03-12|00:02:02.342| [INFO] ........ Inference Time: 1165.0490170791745
Post Proc Phase 3: 100%|######################| 222/222 [00:07<00:00, 28.97it/s]
|2026-03-12|00:03:53.510| [INFO] ........ Total Post Proc Time: 111.16720153298229
|2026-03-12|00:04:12.827| [INFO] ........ Save Time: 19.316651025786996
|2026-03-12|00:04:12.828| [INFO] ................ Finish


Failed: TCGA-AO-A0JI-01A-02-BSB.bfccffbb-dc47-4103-beb9-7799d50af784.svs
Progress: 0/10
---
All done.
Newly processed in this run: 0 files


CompletedProcess(args=['bash', '/data/yujk/hovernet2feature/hovernet-with-feature-extract/run_wsi.sh'], returncode=0)

In [4]:
# Step 1: feature extraction (all WSIs in hovernet_output)
from utils.extract_bulk_features_dinov3 import batch_extract_features

batch_extract_features(
    image_folder=str(IMAGE_FOLDER),
    json_folder=str(JSON_FOLDER),
    mat_folder=str(MAT_FOLDER),
    output_folder=str(FEATURE_ALL_DIR),
    model_name='dinov3',
    pca_components=PCA_COMPONENTS,
    chunk_size=CHUNK_SIZE,
)


Using HuggingFace endpoint: https://hf-mirror.com
✓ DINOv3 available
PyTorch version: 2.6.0+cu124
CUDA available: True
Detected 2 CUDA device(s).
GPU 0: NVIDIA GeForce RTX 4090
GPU 1: NVIDIA GeForce RTX 4090

DinoV3 Feature Extraction Pipeline
Image folder: /data/yujk/hovernet2feature/BiTro/demo_data/output_patches
JSON folder: /data/yujk/hovernet2feature/BiTro/demo_data/hovernet_output
MAT folder: /data/yujk/hovernet2feature/BiTro/demo_data/hovernet_output
Output folder: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Bulk/all
Model: dinov3
PCA components: 128
Chunk size: 100

Checking integrity of previously processed WSIs ...
   • Detected incomplete/corrupted output: TCGA-A7-A0CD-01A-01-TSA.8966c3d7-c0d2-47a4-badf-df9412a1e5f0 -> will reprocess
   • Detected incomplete/corrupted output: TCGA-A7-A0CD-01Z-00-DX1.F045B9C8-049C-41BF-8432-EF89F236D34D -> will reprocess
   • Detected incomplete/corrupted output: TCGA-A8-A06U-01A-01-BS1.1182603b-20e7-4a0e-a686-055841cf5ca4 -> will rep

Extracting Features:   0%|          | 0/355 [00:00<?, ?it/s]

Using HuggingFace endpoint: https://hf-mirror.com
Using HuggingFace endpoint: https://hf-mirror.com
✓ DINOv3 available
PyTorch version: 2.6.0+cu124
✓ DINOv3 available
PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA available: True
Detected 2 CUDA device(s).
GPU 0: NVIDIA GeForce RTX 4090
GPU 1: NVIDIA GeForce RTX 4090
Detected 2 CUDA device(s).
GPU 0: NVIDIA GeForce RTX 4090
GPU 1: NVIDIA GeForce RTX 4090
Using DINOv3 repository: /data/yujk/hovernet2feature/dinov3
Using local DINOv3 weights: /data/yujk/hovernet2feature/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth
Loading DINOv3 ViT-L/16 via torch.hub...
Using DINOv3 repository: /data/yujk/hovernet2feature/dinov3
Using local DINOv3 weights: /data/yujk/hovernet2feature/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth
Loading DINOv3 ViT-L/16 via torch.hub...
✓ Loaded DINOv3 successfully via torch.hub
✓ Loaded DINOv3 successfully via torch.hub


Extracting Features: 100%|██████████| 355/355 [08:53<00:00,  1.50s/it]  



Total cells extracted: 32,217
Applying PCA (target: 128 components)...
PCA completed:
   • Original dimensions: 1024
   • Reduced dimensions: 128
   • Explained variance ratio: 0.8991 (89.91%)
   • Top 10 components:
     PC1: 0.1092 (10.92%)
     PC2: 0.1024 (10.24%)
     PC3: 0.0673 (6.73%)
     PC4: 0.0593 (5.93%)
     PC5: 0.0338 (3.38%)
     PC6: 0.0313 (3.13%)
     PC7: 0.0286 (2.86%)
     PC8: 0.0280 (2.80%)
     PC9: 0.0226 (2.26%)
     PC10: 0.0203 (2.03%)

Results saved:
   • File: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Bulk/all/TCGA-BH-A18H-01A-01-TSA.75dba5a3-f9f5-4ff4-814c-7d2451820e03_features.parquet
   • Cells: 32,217
   • Features per cell: 128
   • File size: 32.97 MB

WSI Progress: 2/10

Processing WSI: TCGA-A7-A0CD-01A-01-TSA.8966c3d7-c0d2-47a4-badf-df9412a1e5f0
   • JSON file: /data/yujk/hovernet2feature/BiTro/demo_data/hovernet_output/json/TCGA-A7-A0CD-01A-01-TSA.8966c3d7-c0d2-47a4-badf-df9412a1e5f0.json
   • MAT dir: not used (WSI-level JSON mode)
T

...


Total cells extracted: 69,959
Applying PCA (target: 128 components)...
PCA completed:
   • Original dimensions: 1024
   • Reduced dimensions: 128
   • Explained variance ratio: 0.8961 (89.61%)
   • Top 10 components:
     PC1: 0.1087 (10.87%)
     PC2: 0.0959 (9.59%)
     PC3: 0.0752 (7.52%)
     PC4: 0.0515 (5.15%)
     PC5: 0.0368 (3.68%)
     PC6: 0.0343 (3.43%)
     PC7: 0.0298 (2.98%)
     PC8: 0.0259 (2.59%)
     PC9: 0.0210 (2.10%)
     PC10: 0.0202 (2.02%)

Results saved:
   • File: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Bulk/all/TCGA-A8-A091-01A-01-BS1.e7e9318d-a627-472a-9f2e-10cc917036d3_features.parquet
   • Cells: 69,959
   • Features per cell: 128
   • File size: 73.62 MB

WSI Progress: 10/10

Processing WSI: TCGA-AO-A0JI-01A-02-BSB.bfccffbb-dc47-4103-beb9-7799d50af784
   • JSON file: /data/yujk/hovernet2feature/BiTro/demo_data/hovernet_output/json/TCGA-AO-A0JI-01A-02-BSB.bfccffbb-dc47-4103-beb9-7799d50af784.json
   • MAT dir: not used (WSI-level JSON mode)
T

Extracting Features:   0%|          | 0/207 [00:00<?, ?it/s]

Using HuggingFace endpoint: https://hf-mirror.com
Using HuggingFace endpoint: https://hf-mirror.com
✓ DINOv3 available
PyTorch version: 2.6.0+cu124
✓ DINOv3 available
PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA available: True
Detected 2 CUDA device(s).
GPU 0: NVIDIA GeForce RTX 4090
GPU 1: NVIDIA GeForce RTX 4090
Detected 2 CUDA device(s).
GPU 0: NVIDIA GeForce RTX 4090
GPU 1: NVIDIA GeForce RTX 4090
Using DINOv3 repository: /data/yujk/hovernet2feature/dinov3
Using local DINOv3 weights: /data/yujk/hovernet2feature/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth
Loading DINOv3 ViT-L/16 via torch.hub...
Using DINOv3 repository: /data/yujk/hovernet2feature/dinov3
Using local DINOv3 weights: /data/yujk/hovernet2feature/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth
Loading DINOv3 ViT-L/16 via torch.hub...
✓ Loaded DINOv3 successfully via torch.hub
✓ Loaded DINOv3 successfully via torch.hub


Extracting Features: 100%|██████████| 207/207 [05:06<00:00,  1.48s/it] 



Total cells extracted: 19,175
Applying PCA (target: 128 components)...
PCA completed:
   • Original dimensions: 1024
   • Reduced dimensions: 128
   • Explained variance ratio: 0.9023 (90.23%)
   • Top 10 components:
     PC1: 0.1244 (12.44%)
     PC2: 0.1004 (10.04%)
     PC3: 0.0618 (6.18%)
     PC4: 0.0553 (5.53%)
     PC5: 0.0400 (4.00%)
     PC6: 0.0338 (3.38%)
     PC7: 0.0321 (3.21%)
     PC8: 0.0245 (2.45%)
     PC9: 0.0215 (2.15%)
     PC10: 0.0202 (2.02%)

Results saved:
   • File: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Bulk/all/TCGA-AO-A0JI-01A-02-BSB.bfccffbb-dc47-4103-beb9-7799d50af784_features.parquet
   • Cells: 19,175
   • Features per cell: 128
   • File size: 19.67 MB

All processing completed!


In [6]:
# Step 2: add cluster_label column to each parquet (required by bulk_graph_construction.py)
cluster_env = os.environ.copy()
cluster_env.update({
    'CLUSTER_SOURCE_DIR': str(FEATURE_ALL_DIR),
    'CLUSTER_N_CLUSTERS': str(N_CLUSTERS),
    'CLUSTER_RANDOM_STATE': str(CLUSTER_RANDOM_STATE),
})
run_cmd(['python', str(PROJECT_ROOT / 'utils' / 'cluster_parquet_features.py')], env=cluster_env, cwd=str(PROJECT_ROOT))



[RUN] python /data/yujk/hovernet2feature/BiTro/utils/cluster_parquet_features.py
[INFO] Clustering parquet files in: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Bulk/all
[INFO] n_clusters=7, random_state=42
[INFO] Found 10 parquet files in /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Bulk/all
[OK] wrote cluster_label: TCGA-A7-A0CD-01A-01-TSA.8966c3d7-c0d2-47a4-badf-df9412a1e5f0_features.parquet
[OK] wrote cluster_label: TCGA-A7-A0CD-01Z-00-DX1.F045B9C8-049C-41BF-8432-EF89F236D34D_features.parquet
[OK] wrote cluster_label: TCGA-A8-A06U-01A-01-BS1.1182603b-20e7-4a0e-a686-055841cf5ca4_features.parquet
[OK] wrote cluster_label: TCGA-A8-A091-01A-01-BS1.e7e9318d-a627-472a-9f2e-10cc917036d3_features.parquet
[OK] wrote cluster_label: TCGA-A8-A09I-01A-01-BS1.b0debb62-86b0-459f-b62b-604bb9f17e5d_features.parquet
[OK] wrote cluster_label: TCGA-AN-A04A-01A-01-BS1.647f0482-49a8-4794-b9c4-5941b14fd1af_features.parquet
[OK] wrote cluster_label: TCGA-AN-A04A-01A-02-BS2.44355b22-95d3-49

CompletedProcess(args=['python', '/data/yujk/hovernet2feature/BiTro/utils/cluster_parquet_features.py'], returncode=0)

In [8]:
# Step 3: split features into train/test
split_env = os.environ.copy()
split_env.update({
    'SPLIT_SOURCE_DIR': str(FEATURE_ALL_DIR),
    'SPLIT_TRAIN_DIR': str(TRAIN_FEATURE_DIR),
    'SPLIT_TEST_DIR': str(TEST_FEATURE_DIR),
})
run_cmd(['python', str(PROJECT_ROOT / 'utils' / 'split_features.py')], env=split_env, cwd=str(PROJECT_ROOT))



[RUN] python /data/yujk/hovernet2feature/BiTro/utils/split_features.py
[INFO] Destination directories ready: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Bulk/train, /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Bulk/test
[INFO] Scanning feature files: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Bulk/all
[INFO] Found 10 feature files

[INFO] Split result:
  Train: 8 files (80.0%)
  Test: 2 files (20.0%)

[INFO] Copying training files to /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Bulk/train...


Copy train:  25%|██▌       | 2/8 [00:00<00:00, 15.01it/s]


[INFO] Copying test files to /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Bulk/test...

[INFO] Split completed!
  Train dir: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Bulk/train (8 files)
  Test dir: /data/yujk/hovernet2feature/BiTro/demo_data/Feature/Bulk/test (2 files)
  Total: 10 files

[INFO] First 5 train files:
  TCGA-AN-A04A-01A-02-BS2.44355b22-95d3-491c-af4c-27225e753d7d_features.parquet
  TCGA-A7-A0CD-01A-01-TSA.8966c3d7-c0d2-47a4-badf-df9412a1e5f0_features.parquet
  TCGA-AO-A0JI-01A-02-BSB.bfccffbb-dc47-4103-beb9-7799d50af784_features.parquet
  TCGA-BH-A18H-01A-01-TSA.75dba5a3-f9f5-4ff4-814c-7d2451820e03_features.parquet
  TCGA-E2-A14P-01A-03-TSC.e301fe1e-c75b-4ed2-b304-16a9d63fd835_features.parquet

[INFO] First 5 test files:
  TCGA-AN-A04A-01A-01-BS1.647f0482-49a8-4794-b9c4-5941b14fd1af_features.parquet
  TCGA-A8-A091-01A-01-BS1.e7e9318d-a627-472a-9f2e-10cc917036d3_features.parquet


Copy test: 100%|██████████| 2/2 [00:00<00:00, 15.63it/s]


CompletedProcess(args=['python', '/data/yujk/hovernet2feature/BiTro/utils/split_features.py'], returncode=0)

In [9]:
# Step 4: construct bulk graphs
from utils.bulk_graph_construction import BulkStaticGraphBuilder

builder = BulkStaticGraphBuilder(
    train_features_dir=str(TRAIN_FEATURE_DIR),
    test_features_dir=str(TEST_FEATURE_DIR),
    bulk_csv_path=str(BULK_CSV_PATH),
    patches_dir=str(PATCHES_DIR),
    wsi_input_dir=str(WSI_INPUT_DIR),
    intra_patch_distance_threshold=INTRA_PATCH_DISTANCE_THRESHOLD,
    inter_patch_k_neighbors=INTER_PATCH_K,
    use_deep_features=True,
    feature_dim=PCA_COMPONENTS,
    max_cells_per_patch=MAX_CELLS_PER_PATCH,
    max_train_slides=MAX_TRAIN_SLIDES,
    max_test_slides=MAX_TEST_SLIDES,
    checkpoint_dir=str(CHECKPOINT_DIR),
)

builder.load_bulk_data()
builder.process_all_slides_new_logic()
metadata = builder.save_graphs_slide_logic(str(GRAPH_OUTPUT_DIR))
builder.save_selected_feature_filenames(str(GRAPH_OUTPUT_DIR))
print('Done graph construction')


=== Loading bulk RNA-seq table ===
Cases: 1095 (multi-column cases: 11; will average columns)
Bulk table shape: (734, 1111)
Valid case IDs: 1095
=== Processing all data (slide-level matching, resume-friendly) ===

=== Initializing checkpoint state (lazy loading to save memory) ===
  Processed train slides found: 0
  Processed test slides found: 0
Found 8 feature files for split 'train'; using first 8
Found 2 feature files for split 'test'; using first 2

Processing train split...


Processing train slides:   0%|          | 0/8 [00:00<?, ?it/s]

Processing slide: TCGA-A7-A0CD-01A-01-TSA.8966c3d7-c0d2-47a4-badf-df9412a1e5f0 (patient: TCGA-A7-A0CD)
  - Number of cells: 116296
  - Slide TCGA-A7-A0CD-01A-01-TSA.8966c3d7-c0d2-47a4-badf-df9412a1e5f0: found 1525 matching patch files
  - Number of matched patch files: 1525
  - Assigning 116296 cells to 1525 patches
  - Assigned 116296/116296 cells to 1502 valid patches


Building intra-patch graphs: 100%|██████████| 1502/1502 [00:27<00:00, 54.42it/s]


  - Successfully built graphs: intra-patch graphs 1502 
  - Inter-patch graph: 24032 edges


...

  - Successfully built graphs: intra-patch graphs 856 
  - Inter-patch graph: 13696 edges


Processing test slides: 100%|██████████| 2/2 [00:40<00:00, 20.04s/it]


  Newly processed slides: 2 

Processing completed:
  - Train slides: 8
  - Test slides: 2

Graph construction statistics:
  - Total slides: 10
  - Slides with graphs: 10
  - Slides keeping only original features: 0
=== Saving graph structures and full cell data (slide level, loaded from checkpoints) ===
train split found 8 checkpoint files; loading and saving them in batches
  Processing train split batch 1/2 (5 files)...
    Batch 1 completed; saving intermediate results...
  Processing train split batch 2/2 (3 files)...
  Merging and saving train final split results...
  Found 1 temporary batches that need merging
  Reloading train full split data from checkpoints...
  Found 8 saved checkpoints for split 'train'


Loading checkpoints (train): 100%|██████████| 8/8 [00:02<00:00,  3.68it/s]


  Starting to save train split data files...
  Saving intra-patch graphs...
  ✓ Intra-patch graphs saved
  Saving inter-patch graph...
  ✓ Inter-patch graph saved
  Saving bulk expression...
  ✓ Bulk expression saved
  Saving cell features...
  ✓ Cell features saved
  Saving cell coordinates...
  ✓ Cell coordinates saved
  Saving cluster labels...
  ✓ Cluster labels saved
  Saving graph status...
  ✓ Graph status saved
  Saving cell mappings...
  ✓ Cell mappings saved
  Saving slide mappings...
  ✓ Slide mappings saved
  Saving metadata...
  ✓ Metadata saved
  All data for the train split has been saved!
train split save completed:
  - Total slides: 8
  - Covered patients: 7
  - Slides with graph data: 8
  - Slides without graph data: 0 (full DINO features kept)
  - Intra-patch graphs: /data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Bulk/bulk_train_intra_patch_graphs.pkl
  - Inter-patch graph: /data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Bulk/bulk_train_inter_patch_graphs.pkl


In [4]:
# Step 5: train bulk model
cmd = [
    'python', 'bulk_model/train.py',
    '--graph-data-dir', str(GRAPH_OUTPUT_DIR),
    '--gene-list-file', str(GENE_LIST_FILE),
    '--features-file', str(FEATURES_TSV_FILE),
    '--tpm-csv-file', str(TPM_CSV_FILE),
    '--batch-size', str(BATCH_SIZE),
    '--graph-batch-size', str(GRAPH_BATCH_SIZE),
    '--num-epochs', str(NUM_EPOCHS),
    '--learning-rate', str(LEARNING_RATE),
    '--weight-decay', str(WEIGHT_DECAY),
]

if USE_LORA:
    cmd += [
        '--use-lora',
        '--lora-r', str(LORA_R),
        '--lora-alpha', str(LORA_ALPHA),
        '--lora-dropout', str(LORA_DROPOUT),
        '--lora-freeze-base',
    ]

run_cmd(cmd, cwd=str(PROJECT_ROOT))



[RUN] python bulk_model/train.py --graph-data-dir /data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Bulk --gene-list-file /data/yujk/hovernet2feature/BiTro/demo_data/Gene/BRCA.txt --features-file /data/yujk/hovernet2feature/BiTro/demo_data/bulk/features.tsv --tpm-csv-file /data/yujk/hovernet2feature/BiTro/demo_data/bulk/tpm-TCGA-BRCA-1000-million.csv --batch-size 2 --graph-batch-size 128 --num-epochs 2 --learning-rate 0.0001 --weight-decay 1e-05 --use-lora --lora-r 8 --lora-alpha 16 --lora-dropout 0.05 --lora-freeze-base


/home/yujk/.local/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /home/yujk/miniconda3/envs/transformer/lib/python3.10/site-packages/libpyg.so: undefined symbol: _ZN5torch8autograd12VariableInfoC1ERKN2at6TensorE
  import torch_geometric.typing
/home/yujk/.local/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/yujk/miniconda3/envs/transformer/lib/python3.10/site-packages/torch_scatter/_scatter_cuda.so: undefined symbol: _ZN2at4_ops16div__Tensor_mode4callERNS_6TensorERKS2_St8optionalIN3c1017basic_string_viewIcEEE
  import torch_geometric.typing
/home/yujk/.local/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: /home/yujk/miniconda3/envs/transformer/lib/python3.10/

PyTorch Geometric version: 2.7.0
=== Environment compatibility check ===
✓ PyTorch: 2.6.0+cu124
✓ CUDA: 12.4
✓ GPU: 2 device(s) - NVIDIA GeForce RTX 4090
✓ cuDNN: 90100
✓ numpy: 1.24.4 (expected: 2.2.4)
✓ pandas: 2.3.3 (expected: 2.2.3)
Error: scikit-learn is not installed
✓ matplotlib: 3.10.1 (expected: 3.10.1)
✓ psutil: 7.2.1 (expected: 7.0.0)
=== Environment check complete ===

=== Optimized: multi-graph batching for better GPU utilization + LoRA ===
=== Loading gene mapping ===
Target genes: 785
Mapped genes: 734
Final gene count: 734
Loading static graph data for split 'train'...
Loading full cell feature tables...
✓ Loaded DINO features for all cells
✓ Loaded spatial coordinates for all cells
✓ Loaded cluster labels for all cells
✓ Loaded per-patient graph status
✓ Loaded cell-to-graph mappings
✓ Loaded slide-to-patient mapping
  - Slides: 8
  - Patients: 7
Loading TPM data (filtered gene set)...
[split] Using 7 patient(s) for split 'train' (leakage protection)
✓ Loaded expressio

/home/yujk/miniconda3/envs/transformer/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Epoch 1/2
  Train Loss: 75429484.250000, Test Loss: 20364184.000000
  *** Saved best model ***

=== Epoch 2 training ===

Batch 0: processing 2 patients
  Patient 1: cell_features=torch.Size([15684, 128]), positions=torch.Size([15684, 2]), has_graphs=True, n_graphs=210
    Graph mode: 210 graphs -> 15684 cells (graph-enhanced)
    Batch processing: 210 graphs -> 15684 cells (single long sequence)
    Patient 1 aggregation: n_cells=15684, aggregated_shape=torch.Size([1, 734]), cluster_loss_sample=0.000000
  Patient 2: cell_features=torch.Size([7501, 128]), positions=torch.Size([7501, 2]), has_graphs=True, n_graphs=135
    Graph mode: 135 graphs -> 7501 cells (graph-enhanced)
    Batch processing: 135 graphs -> 7501 cells (single long sequence)
    Patient 2 aggregation: n_cells=7501, aggregated_shape=torch.Size([1, 734]), cluster_loss_sample=0.000000
  Batch 0 merged predictions: shape=torch.Size([2, 734])
  Loss: 33848568.000000
  Backward pass...
  Backward pass done
    Batch process

CompletedProcess(args=['python', 'bulk_model/train.py', '--graph-data-dir', '/data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Bulk', '--gene-list-file', '/data/yujk/hovernet2feature/BiTro/demo_data/Gene/BRCA.txt', '--features-file', '/data/yujk/hovernet2feature/BiTro/demo_data/bulk/features.tsv', '--tpm-csv-file', '/data/yujk/hovernet2feature/BiTro/demo_data/bulk/tpm-TCGA-BRCA-1000-million.csv', '--batch-size', '2', '--graph-batch-size', '128', '--num-epochs', '2', '--learning-rate', '0.0001', '--weight-decay', '1e-05', '--use-lora', '--lora-r', '8', '--lora-alpha', '16', '--lora-dropout', '0.05', '--lora-freeze-base'], returncode=0)

In [5]:
# Step 6: evaluate bulk model
eval_cmd = [
    'python', 'bulk_model/evaluate.py',
    '--model-path', str(PROJECT_ROOT / 'best_PRAD_lora_model_cluster_norm_attention.pt'),
    '--graph-data-dir', str(GRAPH_OUTPUT_DIR),
    '--gene-list-file', str(GENE_LIST_FILE),
    '--features-file', str(FEATURES_TSV_FILE),
    '--tpm-csv-file', str(TPM_CSV_FILE),
    '--split', 'test',
    '--batch-size', '1',
    '--graph-batch-size', str(GRAPH_BATCH_SIZE),
]

if USE_LORA:
    eval_cmd += [
        '--use-lora',
        '--lora-r', str(LORA_R),
        '--lora-alpha', str(LORA_ALPHA),
        '--lora-dropout', str(LORA_DROPOUT),
        '--lora-freeze-base',
    ]

run_cmd(eval_cmd, cwd=str(PROJECT_ROOT))



[RUN] python bulk_model/evaluate.py --model-path /data/yujk/hovernet2feature/BiTro/best_PRAD_lora_model_cluster_norm_attention.pt --graph-data-dir /data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Bulk --gene-list-file /data/yujk/hovernet2feature/BiTro/demo_data/Gene/BRCA.txt --features-file /data/yujk/hovernet2feature/BiTro/demo_data/bulk/features.tsv --tpm-csv-file /data/yujk/hovernet2feature/BiTro/demo_data/bulk/tpm-TCGA-BRCA-1000-million.csv --split test --batch-size 1 --graph-batch-size 128 --use-lora --lora-r 8 --lora-alpha 16 --lora-dropout 0.05 --lora-freeze-base


/home/yujk/.local/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /home/yujk/miniconda3/envs/transformer/lib/python3.10/site-packages/libpyg.so: undefined symbol: _ZN5torch8autograd12VariableInfoC1ERKN2at6TensorE
  import torch_geometric.typing
/home/yujk/.local/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/yujk/miniconda3/envs/transformer/lib/python3.10/site-packages/torch_scatter/_scatter_cuda.so: undefined symbol: _ZN2at4_ops16div__Tensor_mode4callERNS_6TensorERKS2_St8optionalIN3c1017basic_string_viewIcEEE
  import torch_geometric.typing
/home/yujk/.local/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: /home/yujk/miniconda3/envs/transformer/lib/python3.10/

PyTorch Geometric version: 2.7.0
[EVAL] Using device: cuda:0
=== Loading gene mapping ===
Target genes: 785
Mapped genes: 734
Loading static graph data for split 'test'...
Loading full cell feature tables...
✓ Loaded DINO features for all cells
✓ Loaded spatial coordinates for all cells
✓ Loaded cluster labels for all cells
✓ Loaded per-patient graph status
✓ Loaded cell-to-graph mappings
✓ Loaded slide-to-patient mapping
  - Slides: 2
  - Patients: 2
Loading TPM data (filtered gene set)...
[split] Using 2 patient(s) for split 'test' (leakage protection)
✓ Loaded expression data for 2 patient(s) (split 'test' only)
Sanity check - sample patient expression sum: 1000000.06
✓ Data organized by slide: 2 slides
Dataset stats:
  - Total slides: 2
  - Slides with graphs: 2
  - Slides without graphs: 0 (raw DINO features only)
Loaded split 'test': 2 items
Gene filtering complete. Final gene count: 734
TPM sanity check: patient TCGA-AN-A04A expression sum: 1000000.06
[normalization] Gene normal

CompletedProcess(args=['python', 'bulk_model/evaluate.py', '--model-path', '/data/yujk/hovernet2feature/BiTro/best_PRAD_lora_model_cluster_norm_attention.pt', '--graph-data-dir', '/data/yujk/hovernet2feature/BiTro/demo_data/Graphs/Bulk', '--gene-list-file', '/data/yujk/hovernet2feature/BiTro/demo_data/Gene/BRCA.txt', '--features-file', '/data/yujk/hovernet2feature/BiTro/demo_data/bulk/features.tsv', '--tpm-csv-file', '/data/yujk/hovernet2feature/BiTro/demo_data/bulk/tpm-TCGA-BRCA-1000-million.csv', '--split', 'test', '--batch-size', '1', '--graph-batch-size', '128', '--use-lora', '--lora-r', '8', '--lora-alpha', '16', '--lora-dropout', '0.05', '--lora-freeze-base'], returncode=0)